# Austrian Emissions Projections for the Ministry of Finance

This notebook processes the scenarios as published by the Umweltbundesamt for the **BMF Langfristprognose 2025**.

The data was received as xlsx file from Umweltbundesamt on March 25, 2026.

## Report

https://www.umweltbundesamt.at/studien-reports/publikationsdetail?pub_id=2640

> Treibhausgas-Szenarien für die langfristige Budgetprognose 2025.  
> Wien, 2025  
> Reports, Band 1010  
> ISBN: 978-3-99004-857-3  

See https://www.bmf.gv.at/themen/budget/publikationen/langfristige-budgetprognose.html for more information.

In [ ]:
import nomenclature
import pyam

In [ ]:
from uba_utils import read_uba_file

In [ ]:
df_args = dict(
    model="Umweltbundesamt (2025)",
    region="Austria",
)

In [ ]:
file = "source/REP1010_Tabelle_5_7-12.xlsx"

In [ ]:
energy_args = dict(
    file=file,
    sheet_name="Tab7-12",
    variable_col="Unnamed: 1",
)

In [ ]:
energy_scenario_cols = (
    ("Basis (BMF 2025)", "B,D,H:J", ".1"),
    ("Activity (BMF 2025)", "B,D,K:M", ".2"),
)

In [ ]:
definition = nomenclature.DataStructureDefinition("../../definitions/")

## Emissions

In [ ]:
emission_mapping = {
    "Abfallwirtschaft": None,
    "ESR Gesamt": "Emissions|Kyoto Gases [ESR]",
    "EU-ETS": "Emissions|Kyoto Gases [ETS]",
    "Energie und Industrie": None,
    "Fluorierte Gase": None,
    "Gebäude": None,
    "Gesamt" : "Emissions|Kyoto Gases [excl. LULUCF]",
    "Landwirtschaft": None,
    "Verkehr": None,
}

In [ ]:
df_emission = pyam.concat(
    [
        read_uba_file(
            file=file,
            sheet_name="Tab5",
            skiprows=2,
            variable_col="Unnamed: 0",
            unit="Mt CO2e",
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=emission_mapping,
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in (
            ("Basis (BMF 2025)", "A,C:G", None),
            ("Activity (BMF 2025)", "A,C,H:K", ".1"),
        )
    ]
)

In [ ]:
df_emission.plot()

## Final energy consumption

In [ ]:
prefix = "Final Energy [by Sector]|"

energy_by_sector_mapping = {
    "Verkehr": prefix + "Transportation",
    "Industrie": prefix + "Industry",
    "Haushalte": prefix + "Residential and Commercial|Residential", # noqa
    "Dienstleistungen": prefix + "Residential and Commercial|Commercial and Institutional", # noqa
    "Landwirtschaft": prefix + "Agriculture",
}
energy_by_fuel_mapping = {
    "Kohle": "Final Energy [by Carrier]|Coal",
    "Öl": "Final Energy [by Carrier]|Oil",
    "Gas": "Final Energy [by Carrier]|Natural Gas",
    "Biomasse": "Final Energy [by Carrier]|Biomass",
    "Abfall": "Final Energy [by Carrier]|Waste",
    "Wasserstoff; e-Fuels": "Final Energy [by Carrier]|Hydrogen and E-Fuels",
    "Strom": "Final Energy [by Carrier]|Electricity",
    "Wärme": "Final Energy [by Carrier]|District Heat and Ambient Heat",
}

In [ ]:
df_energy = pyam.concat(
    [
        pyam.concat(
            [
                read_uba_file(
                    **energy_args,
                    scenario=scenario,
                    usecols=usecols,
                    col_suffix=col_suffix,
                    variable_mapping=energy_by_sector_mapping,
                    skiprows=2,
                    nrows=6,
                    unit="PJ",
                    df_args=df_args,
                ),
                read_uba_file(
                    **energy_args,
                    scenario=scenario,
                    usecols=usecols,
                    col_suffix=col_suffix,
                    variable_mapping=energy_by_fuel_mapping,
                    skiprows=11,
                    nrows=9,
                    unit="PJ",
                    df_args=df_args,
                ),
            ]
        )
        for scenario, usecols, col_suffix in energy_scenario_cols
    ]
)

In [ ]:
df_energy.aggregate(
    "Final Energy [by Sector]|Residential and Commercial",
    append=True
)
df_energy.aggregate(
    "Final Energy",
    components=df_energy.filter(variable="Final Energy [by Sector]*", level=1).variable,
    append=True
)

In [ ]:
df_energy.check_aggregate(
    "Final Energy",
    components=df_energy.filter(variable="Final Energy [by Carrier]*").variable,
)

In [ ]:
df_energy.convert_unit("PJ", "TJ", inplace=True)

## Electricity consumption by sector

In [ ]:
prefix = "Final Energy [by Sector]|"

electricity_consumption_by_sector_mapping = {
    "Verkehr": prefix + "Transportation|Electricity",
    "Industrie": prefix + "Industry|Electricity",
    "Gebäude": prefix + "Residential and Commercial|Electricity",
    "Landwirtschaft": prefix + "Agriculture|Electricity",
}

In [ ]:
df_electricity_consumption = pyam.concat(
    [
        read_uba_file(
            **energy_args,
            scenario=scenario,
            usecols=usecols,
            col_suffix=col_suffix,
            variable_mapping=electricity_consumption_by_sector_mapping,
            skiprows=23,
            nrows=7,
            unit="PJ",
            df_args=df_args,
        )
        for scenario, usecols, col_suffix in energy_scenario_cols
    ]
)

In [ ]:
df_electricity_consumption.convert_unit("PJ", "TJ", inplace=True)

## Power generation

In [ ]:
# note that there are whitespaces in the xlsx file
electricity_by_source_mapping = {
    "fossil": "Secondary Energy|Electricity|Natural Gas",
    "Wasserkraft ": "Secondary Energy|Electricity|Hydro",
    "Biomasse": "Secondary Energy|Electricity|Biomass",
    "Umgebungswärme etc. ": "Secondary Energy|Electricity|Geothermal",
    "Photovoltaik": "Secondary Energy|Electricity|Solar",
    "Wind ": "Secondary Energy|Electricity|Wind",
}

In [ ]:
df_electricity = pyam.concat(
    [
        pyam.concat(
            [
                read_uba_file(
                    **energy_args,
                    scenario=scenario,
                    usecols=usecols,
                    col_suffix=col_suffix,
                    variable_mapping=electricity_by_source_mapping,
                    skiprows=35,
                    nrows=7,
                    unit="PJ",
                    df_args=df_args,
                ),
                read_uba_file(
                    **energy_args,
                    scenario=scenario,
                    usecols=usecols,
                    col_suffix=col_suffix,
                    variable_mapping={"Nettoimporte in TWh": "Net Imports|Electricity"},
                    skiprows=35,
                    nrows=10,
                    unit="TWh",
                    df_args=df_args,
                ),
            ],
        )
        for scenario, usecols, col_suffix in energy_scenario_cols
    ]
)

In [ ]:
df_electricity.aggregate("Secondary Energy|Electricity", append=True)

In [ ]:
df_electricity.convert_unit("PJ", "TWh", inplace=True)

## Concatenate, add meta indicators, and export

In [ ]:
df = pyam.concat(
    [
        df_emission, df_energy, df_electricity_consumption, df_electricity
    ]
)

In [ ]:
df.set_meta(
    name="Citation",
    meta="Treibhausgas-Szenarien für die langfristige Budgetprognose, Report 1010 (UBA, 2025)", # noqa
)

In [ ]:
df

In [ ]:
definition.validate(df)

In [ ]:
df.to_excel("uba_bmf_2025.xlsx")